# PyroPredict — Data Setup & Exploratory Data Analysis

**CMPE 258 Deep Learning** | Spring 2026 | SJSU

This notebook downloads the D-Fire wildfire smoke/fire dataset, validates it,
runs exploratory analysis, and saves everything to Google Drive for training.

**Runtime:** CPU is fine (no GPU needed)  
**Time:** ~5–10 minutes total

---

### What you need before running

1. A **free Kaggle account** → [kaggle.com](https://www.kaggle.com)  
2. Your **Kaggle username** (shown on your profile page) and **API token**:  
   Kaggle → Settings → API → *Create New Token* → copy the token string  
3. That's it. Paste them into the config cell below and run all cells top to bottom.

## 0 — Install dependencies & mount Drive

In [ ]:
!pip install -q ultralytics opencv-python-headless seaborn pyyaml kaggle

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT = '/content/drive/MyDrive/PyroPredict'
os.makedirs(DRIVE_PROJECT, exist_ok=True)
print(f'Drive project folder: {DRIVE_PROJECT}')

## 1 — Kaggle credentials & download D-Fire dataset

Paste your Kaggle **username** and **API token** in the cell below, then run it.

In [ ]:
# ┌──────────────────────────────────────────────────────┐
# │  PASTE YOUR KAGGLE USERNAME AND TOKEN BELOW          │
# └──────────────────────────────────────────────────────┘
KAGGLE_USERNAME = "your_username_here"   # ← replace with your Kaggle username
KAGGLE_TOKEN    = "your_token_here"      # ← replace with the token you copied

import json

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json = os.path.join(kaggle_dir, 'kaggle.json')
with open(kaggle_json, 'w') as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_TOKEN}, f)
os.chmod(kaggle_json, 0o600)
print(f'Kaggle credentials saved for user: {KAGGLE_USERNAME}')

In [ ]:
%%time

DATA_DIR = '/content/dfire'
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d sayedgamal99/smoke-fire-detection-yolo -p /content/
!unzip -q /content/smoke-fire-detection-yolo.zip -d {DATA_DIR}
!rm -f /content/smoke-fire-detection-yolo.zip

print('\nDownloaded contents:')
for root, dirs, fls in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in fls[:5]:
            print(f'{indent}  {f}')
        if len(fls) > 5:
            print(f'{indent}  ... and {len(fls)-5} more files')

## 2 — Auto-detect dataset layout & build dataset.yaml

In [ ]:
from pathlib import Path
import yaml

data_root = Path(DATA_DIR)

# Auto-detect the directory structure
# D-Fire Kaggle layout is typically:
#   dfire/train/images, dfire/train/labels
#   dfire/valid/images, dfire/valid/labels  (or val)
#   dfire/test/images,  dfire/test/labels

def find_split_dirs(root):
    """Walk the tree and find image/label split directories."""
    splits = {}
    for split_name in ['train', 'valid', 'val', 'test']:
        for candidate in [
            root / split_name / 'images',
            root / 'images' / split_name,
        ]:
            if candidate.is_dir() and any(candidate.iterdir()):
                canon = 'val' if split_name == 'valid' else split_name
                lbl_dir = candidate.parent / 'labels' if (candidate.parent / 'labels').is_dir() else root / split_name / 'labels'
                splits[canon] = {'images': candidate, 'labels': lbl_dir}
                break
    # If nothing found at depth 1, search one level deeper
    if not splits:
        for child in root.iterdir():
            if child.is_dir():
                deeper = find_split_dirs(child)
                if deeper:
                    return deeper
    return splits

splits = find_split_dirs(data_root)

if not splits:
    raise FileNotFoundError(
        f'Could not find train/val/test image directories under {data_root}. '
        f'Contents: {[p.name for p in data_root.iterdir()]}'
    )

print('Detected splits:')
for name, dirs in splits.items():
    n_img = len(list(dirs['images'].glob('*')))
    n_lbl = len(list(dirs['labels'].glob('*.txt')))
    print(f"  {name:>5s}: {n_img:>6,} images, {n_lbl:>6,} labels  [{dirs['images']}]")

In [ ]:
# Determine class names from the label files
from collections import Counter

class_ids_found = Counter()
sample_split = list(splits.values())[0]
for lbl_path in list(sample_split['labels'].glob('*.txt'))[:2000]:
    for line in lbl_path.read_text().splitlines():
        parts = line.strip().split()
        if parts:
            class_ids_found[int(parts[0])] += 1

print(f'Class IDs found (from first 2000 train labels): {dict(class_ids_found)}')

# D-Fire classes: 0=fire, 1=smoke  (or 0=smoke, 1=fire depending on version)
# We detect the mapping from the counts — smoke is more common in D-Fire
sorted_ids = sorted(class_ids_found.keys())
if len(sorted_ids) == 2:
    # The class with more boxes in D-Fire is typically fire (14,692 vs 11,865)
    CLASS_NAMES = {sorted_ids[0]: 'fire', sorted_ids[1]: 'smoke'}
elif len(sorted_ids) == 1:
    CLASS_NAMES = {sorted_ids[0]: 'smoke'}
else:
    CLASS_NAMES = {i: f'class_{i}' for i in sorted_ids}

print(f'Class mapping: {CLASS_NAMES}')
print()
print('NOTE: Verify this mapping matches the dataset README.')
print('If fire=0, smoke=1 is wrong, swap them in the cell below before continuing.')

In [ ]:
# ┌─────────────────────────────────────────────────────────┐
# │  EDIT HERE if the auto-detected class mapping is wrong  │
# └─────────────────────────────────────────────────────────┘

# CLASS_NAMES = {0: 'smoke', 1: 'fire'}   # uncomment & fix if needed

NC = len(CLASS_NAMES)
print(f'Final class mapping ({NC} classes): {CLASS_NAMES}')

In [ ]:
# Write dataset.yaml (needed by YOLO training)
dataset_yaml_path = data_root / 'dataset.yaml'

yaml_content = {
    'path': str(data_root.resolve()),
    'train': str(splits['train']['images'].relative_to(data_root)),
    'val':   str(splits['val']['images'].relative_to(data_root)),
    'test':  str(splits['test']['images'].relative_to(data_root)) if 'test' in splits else '',
    'nc':    NC,
    'names': list(CLASS_NAMES.values()),
}

with open(dataset_yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)

print(f'dataset.yaml written to {dataset_yaml_path}')
print()
print(dataset_yaml_path.read_text())

## 3 — Validate labels

Check every split for: missing labels, orphan labels, malformed lines, class counts.

In [ ]:
import numpy as np

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

def validate_split(img_dir, lbl_dir, split_name):
    img_stems = {p.stem for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS}
    lbl_stems = {p.stem for p in lbl_dir.glob('*.txt')}

    missing = img_stems - lbl_stems
    orphan  = lbl_stems - img_stems

    class_counts = Counter()
    total_boxes = 0
    malformed = 0

    for lbl in lbl_dir.glob('*.txt'):
        for line in lbl.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                malformed += 1
                continue
            try:
                cls_id = int(parts[0])
                vals = [float(v) for v in parts[1:]]
                class_counts[cls_id] += 1
                total_boxes += 1
            except ValueError:
                malformed += 1

    print(f'\n── {split_name} ──')
    print(f'  Images:         {len(img_stems):>7,}')
    print(f'  Label files:    {len(lbl_stems):>7,}')
    print(f'  Bounding boxes: {total_boxes:>7,}')
    print(f'  Missing labels: {len(missing):>7}  (images with no label)')
    print(f'  Orphan labels:  {len(orphan):>7}  (labels with no image)')
    print(f'  Malformed:      {malformed:>7}')
    print(f'  Per class:      {dict(class_counts)}')
    return {
        'images': len(img_stems), 'labels': len(lbl_stems),
        'boxes': total_boxes, 'classes': dict(class_counts),
        'missing': len(missing), 'orphan': len(orphan),
    }

all_stats = {}
for name, dirs in splits.items():
    all_stats[name] = validate_split(dirs['images'], dirs['labels'], name)

## 4 — Visualise annotated samples

In [ ]:
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

COLORS = {0: (0.93, 0.35, 0.18), 1: (1.0, 0.6, 0.0)}

def draw_boxes(img_path, lbl_path, ax):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls = int(parts[0])
            xc, yc, bw, bh = [float(v) for v in parts[1:]]
            x1 = (xc - bw/2) * w
            y1 = (yc - bh/2) * h
            color = COLORS.get(cls, (0.3, 0.7, 0.9))
            rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                     linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            label = CLASS_NAMES.get(cls, str(cls))
            ax.text(x1, y1-4, label, fontsize=8, fontweight='bold', color='white',
                    bbox=dict(facecolor=color, alpha=0.7, pad=1, edgecolor='none'))
    ax.set_axis_off()
    ax.set_title(img_path.name, fontsize=7)

rng = np.random.RandomState(42)
train_imgs = sorted(splits['train']['images'].glob('*'))
train_imgs = [p for p in train_imgs if p.suffix.lower() in IMG_EXTS]
chosen = rng.choice(len(train_imgs), size=min(12, len(train_imgs)), replace=False)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for i, idx in enumerate(chosen):
    img_p = train_imgs[idx]
    lbl_p = splits['train']['labels'] / f'{img_p.stem}.txt'
    draw_boxes(img_p, lbl_p, axes.flat[i])
fig.suptitle('Training samples with bounding boxes', fontsize=14)
fig.tight_layout()
fig.savefig(f'{DRIVE_PROJECT}/eda_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {DRIVE_PROJECT}/eda_samples.png')

## 5 — Class distribution per split

In [ ]:
import seaborn as sns
import pandas as pd

rows = []
for split_name, stats in all_stats.items():
    for cls_id, count in stats['classes'].items():
        rows.append({'split': split_name, 'class': CLASS_NAMES.get(cls_id, str(cls_id)), 'boxes': count})

df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=df, x='split', y='boxes', hue='class',
            palette=['#ee5a2e', '#ff9900'], ax=ax)
ax.set_title('Bounding-box count per class per split')
ax.set_ylabel('Number of bounding boxes')
for container in ax.containers:
    ax.bar_label(container, fmt='{:,.0f}', fontsize=8)
fig.tight_layout()
fig.savefig(f'{DRIVE_PROJECT}/eda_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 — Image resolution & bounding-box size distributions

In [ ]:
widths, heights, box_ws, box_hs, box_classes = [], [], [], [], []

sample_imgs = rng.choice(train_imgs, size=min(2000, len(train_imgs)), replace=False)

for img_path in sample_imgs:
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    h, w = img.shape[:2]
    widths.append(w)
    heights.append(h)
    lbl_path = splits['train']['labels'] / f'{img_path.stem}.txt'
    if not lbl_path.exists():
        continue
    for line in lbl_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls = int(parts[0])
        bw_px = float(parts[3]) * w
        bh_px = float(parts[4]) * h
        box_ws.append(bw_px)
        box_hs.append(bh_px)
        box_classes.append(CLASS_NAMES.get(cls, str(cls)))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(widths, bins=30, color='#3b82f6', alpha=0.7, label='width')
axes[0].hist(heights, bins=30, color='#f59e0b', alpha=0.7, label='height')
axes[0].set_title('Image resolution distribution')
axes[0].set_xlabel('Pixels')
axes[0].legend()

axes[1].hist(box_ws, bins=50, color='#ee5a2e', alpha=0.7)
axes[1].set_title('Box width (px)')
axes[1].set_xlabel('Pixels')

axes[2].hist(box_hs, bins=50, color='#ff9900', alpha=0.7)
axes[2].set_title('Box height (px)')
axes[2].set_xlabel('Pixels')

fig.tight_layout()
fig.savefig(f'{DRIVE_PROJECT}/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Box width vs height scatter
fig, ax = plt.subplots(figsize=(7, 6))
df_box = pd.DataFrame({'width': box_ws, 'height': box_hs, 'class': box_classes})
for cls_name, color in zip(CLASS_NAMES.values(), ['#ee5a2e', '#ff9900']):
    subset = df_box[df_box['class'] == cls_name]
    ax.scatter(subset['width'], subset['height'], alpha=0.2, s=6, color=color, label=cls_name)
ax.set_xlabel('Box width (px)')
ax.set_ylabel('Box height (px)')
ax.set_title('Bounding-box size: width vs height')
ax.legend()
fig.tight_layout()
fig.savefig(f'{DRIVE_PROJECT}/eda_box_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 — Split summary

In [ ]:
summary_rows = []
for name, stats in all_stats.items():
    summary_rows.append({
        'Split': name,
        'Images': stats['images'],
        'Labels': stats['labels'],
        'Boxes': stats['boxes'],
        **{CLASS_NAMES.get(k, str(k)): v for k, v in stats['classes'].items()},
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(5, 5))
sizes = [s['images'] for s in all_stats.values()]
labels = list(all_stats.keys())
colors = ['#3b82f6', '#f59e0b', '#10b981']
ax.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors[:len(labels)],
       startangle=90, textprops={'fontsize': 11})
ax.set_title('Dataset split proportions')
fig.tight_layout()
fig.savefig(f'{DRIVE_PROJECT}/eda_split_pie.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 — Negative samples (images with no annotations)

D-Fire includes ~9,800 negative images. Let's verify and visualise a few.

In [ ]:
neg_count = 0
pos_count = 0
for img_path in splits['train']['images'].iterdir():
    if img_path.suffix.lower() not in IMG_EXTS:
        continue
    lbl = splits['train']['labels'] / f'{img_path.stem}.txt'
    if not lbl.exists() or lbl.stat().st_size == 0:
        neg_count += 1
    else:
        pos_count += 1

print(f'Training set: {pos_count:,} positive images, {neg_count:,} negative images')
print(f'Negative ratio: {neg_count / (pos_count + neg_count):.1%}')

## 9 — Save everything to Google Drive

In [ ]:
import json, shutil

# Save EDA stats
eda_report = {
    'class_names': CLASS_NAMES,
    'splits': {name: stats for name, stats in all_stats.items()},
    'dataset_yaml_path': str(dataset_yaml_path),
    'data_root': str(data_root),
}

with open(f'{DRIVE_PROJECT}/eda_report.json', 'w') as f:
    json.dump(eda_report, f, indent=2)

# Copy dataset.yaml to Drive for Phase 3
shutil.copy2(dataset_yaml_path, f'{DRIVE_PROJECT}/dataset.yaml')

print('Saved to Google Drive:')
for p in sorted(Path(DRIVE_PROJECT).glob('*')):
    print(f'  {p.name}')

print(f'\n{"="*60}')
print('EDA COMPLETE')
print(f'{"="*60}')
print(f'\nDataset root (for Phase 3 notebook): {data_root}')
print(f'dataset.yaml: {dataset_yaml_path}')
print(f'Drive backup: {DRIVE_PROJECT}')

---

## Summary for Phase 3

Copy the **Dataset root** path and **dataset.yaml** path printed above.
You will paste them into the Phase 3 training notebook.

Also share these numbers so the training configs can be tailored:
- Total images per split (train / val / test)
- Number of classes
- Class names and box counts

**Next →** Open `02_train_baselines.ipynb` in Colab with a **GPU runtime**.